# 02 · Fetch & explore (against fixtures)

The fetchers normalize raw Microsoft Graph JSON into typed models. Here we run
those same mappers on the committed **synthetic fixtures**, so the output is
fully reproducible without a live tenant.

In [ ]:
import sys, json
from pathlib import Path

# Repo root = parent of the notebooks/ directory.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
FIXTURES = REPO / "tests" / "fixtures"
print("repo:", REPO)
print("fixtures:", [p.name for p in FIXTURES.glob("*.json")])

In [ ]:
from m365_review.core.models import Organization, SubscribedSku, User, TenantData

org = Organization.from_graph(json.loads((FIXTURES / "organization.json").read_text())["value"][0])
skus = [SubscribedSku.from_graph(s) for s in json.loads((FIXTURES / "subscribedSkus.json").read_text())["value"]]
users = [User.from_graph(u) for u in json.loads((FIXTURES / "users.json").read_text())["value"]]

data = TenantData(
    organization=org, tenant_id=org.id, skus=skus, users=users,
    sign_in_activity_available=True,
)
print(f"{org.display_name}: {len(skus)} SKUs, {len(users)} users")

## Subscribed SKUs — purchased vs. assigned

In [ ]:
for s in skus:
    print(f"{s.sku_part_number:<20} purchased={s.prepaid_enabled:>3} assigned={s.consumed_units:>3} slack={s.slack:>3}")

## Users — enabled, type, licenses, last sign-in

In [ ]:
for u in users:
    last = u.effective_last_sign_in.date().isoformat() if u.effective_last_sign_in else "never"
    print(f"{(u.user_principal_name or ''):<40} enabled={u.account_enabled!s:<5} "
          f"type={u.user_type or '':<6} licensed={u.is_licensed!s:<5} last_sign_in={last}")